# 04. 워크보드와 TUI: 여러 작업을 쌓아 두고 검증된 것만 완료하기

01~03에서는 `syncmill run`으로 작업을 **하나씩** 맡겼습니다. 실제 프로젝트에서는 할 일이 여러 개이고, 순서가 있고, 누가
무엇을 했는지·어디서 막혔는지·정말 끝났는지를 기록해 둬야 합니다. 이 노트북에서는 그것을 맡는 **워크보드(board)**를 써 보고,
같은 내용을 화면으로 다루는 **TUI**를 소개합니다.

**이런 분께 맞습니다**

- 할 일 목록을 에이전트들에게 차례로 처리시키고 싶은 분
- "에이전트가 끝났다고 했는데 정말 끝난 건가?"를 기록으로 구분하고 싶은 분
- 팀에서 정한 규칙을 에이전트가 매번 다시 따지지 않게 하고 싶은 분

**이 노트북에서 해 보는 것**

| 장 | 하는 일 | 끝나면 알게 되는 것 |
|---|---|---|
| 0 | 워크보드가 필요한 이유 | `run`과 무엇이 다른지, 작업 상태의 흐름 |
| 1~2 | 보드 켜기, 작업 등록 | 목표(goal)와 의존 관계 |
| 3 | 팀 결정 기록 | 결정이 프롬프트에 들어가는 방식 |
| 4 | 보드에서 실행 | 에이전트가 받는 프롬프트, "완료" 요청이 `review`로 바뀌는 이유 |
| 5 | 결과 적용과 사람의 완료 처리 | 보드 상태와 원본 적용은 별개라는 것 |
| 6 | 검사가 있으면 자동 완료 | 검증된 완료의 기준 |
| 7 | 실패한 작업 | `blocked`와 사람 확인 표시 |
| 8 | 팀 상태 지표 | `board stats` 읽는 법 |
| 9 | TUI | 터미널 화면에서 같은 일을 하는 방법 |

**비용과 안전**: 01과 같은 샌드박스에서 가짜 에이전트로 실행합니다. 실제 AI 호출은 없습니다. 소요 시간은 약 10분입니다.

## 0. 왜 워크보드가 필요한가요?

`syncmill run`은 **한 번 맡기고 결과를 받는** 도구입니다. 실행이 끝나면 "다음에 뭘 해야 하지?", "이건 왜 멈췄지?" 같은
정보는 어디에도 남지 않습니다.

워크보드는 작업마다 **지속되는 기록**을 남깁니다.

| 기록 | 뜻 |
|---|---|
| 목표(goal) | 이 작업이 속한 큰 목표(예: `textstats`, `backend`) |
| 상태(state) | 지금 어디까지 왔는지(아래 흐름) |
| 의존(depends) | 먼저 끝나야 하는 작업 |
| 담당(owner) | 마지막으로 처리한 에이전트 |
| 증거(evidence) | 완료를 뒷받침하는 검증 기록(검사 통과, 사람 확인) |
| 인계 메모(handoff) | 다음에 이 작업을 잡는 사람·에이전트에게 남기는 메모 |
| 이력(history) | 상태가 언제, 누구 때문에 바뀌었는지 |

작업 상태는 이렇게 흐릅니다.

```text
planned ──(실행 시작)──▶ running ──▶ review ──(검증 또는 사람 확인)──▶ done
                                   └─▶ blocked (실패, 사람 확인 필요)
```

핵심 규칙: **`done`은 "끝났다"가 아니라 "끝났고 검증됐다"**입니다. 에이전트가 "다 했어요"라고 해도, 그 실행에서 품질 게이트를
통과했거나 사람이 확인(sign-off)하기 전에는 `review`에 머뭅니다.

워크보드는 공유 메모리(03)와 다른 별도 저장소(SQLite)를 씁니다. **상태는 보드에, 지식은 메모리에** 둡니다.

## 1. 준비: 보드 켜기

워크보드는 **기본적으로 꺼져 있습니다.** 프로젝트에서 쓰려면 `syncmill.toml`에 `[board] enabled = true`를 넣습니다
(한 번만 켜려면 환경 변수 `SYNCMILL_BOARD__ENABLED=true`). 팀이 함께 쓰는 설정이므로 커밋합니다.

In [1]:
import json
import re

from _sandbox import Sandbox

box = Sandbox.create()
box.init_project()

toml_path = box.repo / "syncmill.toml"
toml_path.write_text(toml_path.read_text(encoding="utf-8") + "\n[board]\nenabled = true\n", encoding="utf-8")
box.git("add", "syncmill.toml")
box.git("commit", "-qm", "enable the work board")
print(toml_path.read_text(encoding="utf-8"))


def item_id(result):
    """The id from a `board add` result (`[syncmill] added item <id>`)."""
    return re.search(r"added item ([0-9a-f-]{36})", result.stdout).group(1)


def board_block(**fields):
    """The optional block an agent may append to report a board update."""
    return "\n```syncmill-board\n" + json.dumps(fields, ensure_ascii=False) + "\n```"

config_version = 2
repo_root = "."
default_strategy = "route"

[project]
id = "5a8bb136-069e-4da3-874c-dc42d84c290c"
memory_dir = ".syncmill/memory"

[gate]
commands = []

[board]
enabled = true



## 2. 작업 등록: `board add`

두 작업을 등록합니다. 두 번째 작업은 첫 번째가 **끝나야(`done`)** 시작할 수 있도록 `--depends-on`으로 묶습니다.

In [2]:
first = box.sm("board", "add", "textstats.py에 count_words(text)를 구현해 줘", "--goal", "textstats")
impl_id = item_id(first)

second = box.sm("board", "add", "README에 count_words 사용 예시를 추가해 줘",
                "--goal", "textstats", "--depends-on", impl_id)
docs_id = item_id(second)

[syncmill] added item cc49cf9f-6440-494b-a820-a5d49ea26985
id:       cc49cf9f-6440-494b-a820-a5d49ea26985
state:    planned
goal:     textstats
task:     textstats.py에 count_words(text)를 구현해 줘
owner:    -
[syncmill] added item ec54694f-1d25-4ce3-aaee-fe091a16c757
id:       ec54694f-1d25-4ce3-aaee-fe091a16c757
state:    planned
goal:     textstats
task:     README에 count_words 사용 예시를 추가해 줘
owner:    -
depends:  cc49cf9f-6440-494b-a820-a5d49ea26985


**출력 읽는 법 — `board add`**

| 줄 | 뜻 |
|---|---|
| `added item <id>` | 작업 id. 이후 `show`·`update`·`--depends-on`에 씀 |
| `state: planned` | 등록만 된 상태 |
| `goal:` | 목표. 같은 목표끼리 묶어 실행·조회할 수 있음 |
| `owner: -` | 아직 처리한 에이전트 없음 |
| `depends:` | 먼저 끝나야 하는 작업의 id |

목록으로 보면 이렇습니다.

In [3]:
box.sm("board", "list")

  cc49cf9f-6440-494b-a820-a5d49ea26985  planned   [textstats] textstats.py에 count_words(text)를 구현해 줘
  ec54694f-1d25-4ce3-aaee-fe091a16c757  planned   [textstats] README에 count_words 사용 예시를 추가해 줘  (deps=1)


<syncmill board: exit 0>

한 줄이 작업 하나이고 `id  상태  [목표] 할 일` 순서입니다. `(deps=1)`은 **등록된 선행 작업이 1개** 있다는 뜻입니다. 선행 작업이
끝났는지는 이 표시로 알 수 없으니, 그 작업의 상태(`done`인지)를 확인하세요.

## 3. 팀 결정 기록: `board decide`

"표준 라이브러리만 쓴다", "REST만 쓰고 GraphQL은 쓰지 않는다"처럼 **한 번 정한 규칙**을 에이전트가 매번 다시 따지지 않게
하려면 결정으로 기록합니다. `--scope`로 목표를 지정하면 그 목표의 작업에만 적용됩니다.

In [4]:
box.sm("board", "decide", "표준 라이브러리만 사용한다", "--scope", "textstats")
box.sm("board", "decisions")

[syncmill] recorded decision 42d6c01c-8027-4ef5-b7a0-e857e7bcdd16
  42d6c01c-8027-4ef5-b7a0-e857e7bcdd16  [textstats] 표준 라이브러리만 사용한다
  42d6c01c-8027-4ef5-b7a0-e857e7bcdd16  [textstats] 표준 라이브러리만 사용한다


<syncmill board: exit 0>

기록된 결정은 해당 목표의 보드 실행 프롬프트에 **검색 없이 통째로** 들어갑니다(4장에서 확인). 공유 메모리처럼 관련도로 골라
붙이는 것이 아니라서 **빠지지 않습니다.** 결정을 바꿀 때는 `--supersedes <결정 id>`로 새 결정을 기록하면 예전 결정은 은퇴합니다.

## 4. 보드에서 실행하기: `board run`

`board run`은 **지금 처리할 수 있는 다음 작업 하나**를 골라(선행 작업이 끝났고, 사람 확인이 필요하지 않은 것 중 먼저 등록된 것)
`run`과 같은 방식으로 실행하고, 결과를 보드에 기록합니다. `run`과 같은 `-s`·`--agent`·`--team` 옵션을 받습니다.

에이전트는 끝낼 때 보드 상태를 **제안하는 블록**을 답변 끝에 붙일 수 있습니다. 이번 연출에서 가짜 codex는 구현을 마치고
"완료(`done`)"를 요청합니다.

In [5]:
IMPL = "def count_words(text):\n    return len(text.split())\n"
box.set_scenario({"codex": {
    "files": {"textstats.py": IMPL},
    "message": "Implemented count_words."
               + board_block(state="done", handoff="count_words는 split() 기반입니다", outputs=["textstats.py"]),
}})
start = len(box.prompts())
box.sm("board", "run", "--agent", "codex")


[codex] Completed in 0.35s
[syncmill] Files changed: textstats.py

Implemented count_words.
```syncmill-board
{"state": "done", "handoff": "count_words는 split() 기반입니다", "outputs": ["textstats.py"]}
```


<syncmill board: exit 0>

에이전트가 실제로 받은 프롬프트를 봅시다.

In [6]:
print(box.prompts()[-1].read_text(encoding="utf-8"))

You are working on a tracked work-board item.
Goal: textstats

Team decisions you MUST respect (do not re-litigate):
- 표준 라이브러리만 사용한다

Task: textstats.py에 count_words(text)를 구현해 줘

When you finish, you MAY append ONE fenced block reporting the board update (every field optional):
```syncmill-board
{"state": "review", "handoff": "<note for whoever picks this up next>", "outputs": ["<artifact or file you produced>"], "human_required": false}
```
Use "state": "blocked" with a "blocked_reason" if you cannot proceed, or "human_required": true if a human must step in. Requesting "done" only takes effect if the work is independently verified (a passing quality gate or a human sign-off); otherwise it is recorded as "review".


**출력 읽는 법 — 보드 실행 프롬프트**

- `You are working on a tracked work-board item.` / `Goal: textstats` — 보드 작업이라는 것과 목표.
- `Team decisions you MUST respect` — 3장에서 기록한 결정이 **그대로** 들어갔습니다.
- `Task:` — 등록한 할 일.
- 마지막 문단 — 보드 상태를 알리는 블록(```` ```syncmill-board ````)의 형식과 규칙. 특히 "`done` 요청은 독립적으로 검증된 경우에만
  효력이 있고, 아니면 `review`로 기록된다"고 에이전트에게도 알려 줍니다.

이제 보드에 무엇이 기록됐는지 봅시다.

In [7]:
shown = box.sm("board", "show", impl_id).stdout
state = re.search(r"(?m)^state:\s+(\w+)", shown).group(1)
print("상태:", state)
assert state == "review"

id:       cc49cf9f-6440-494b-a820-a5d49ea26985
state:    review
goal:     textstats
task:     textstats.py에 count_words(text)를 구현해 줘
owner:    codex
outputs:  textstats.py
handoff:  count_words는 split() 기반입니다
history:
  · → planned  [human] — created
  planned → running  [ce3ec018-06d8-4a91-96aa-32385d8f0f89] — claimed
  running → review  [codex] — agent requested done without verification — clamped to review
상태: review


**출력 읽는 법 — `board show`**

- `state: review` — 에이전트는 `done`을 요청했지만 **`review`로 기록됐습니다.** 이번 실행에는 품질 게이트가 없어서 완료를
  뒷받침할 검증이 없었기 때문입니다.
- `owner: codex`, `outputs: textstats.py`, `handoff: …` — 에이전트가 블록으로 알린 내용이 기록됐습니다.
- `history`의 마지막 줄 `agent requested done without verification — clamped to review` — 왜 `review`가 됐는지가 이력에 남습니다.
- `planned → running [<id>] — claimed`의 대괄호 안 값은 **이 실행의 run id**입니다(다음 장에서 씁니다).

**의존 관계도 확인해 봅시다.** 두 번째 작업은 첫 번째가 `done`이 아니므로 아직 실행할 수 없습니다.

In [8]:
waiting = box.sm("board", "run", "--agent", "codex")
assert "nothing to run" in waiting.stdout

[syncmill] nothing to run — no dispatchable item
[10/14/26 21:15:23] INFO     board: purged 1 stale board-run  supervisor.py:1307
                             namespace(s)                                       


`nothing to run — no dispatchable item` — 처리할 수 있는 작업이 없다는 뜻입니다. 남은 작업은 선행 작업이 끝나기를 기다리고 있습니다.

## 5. 결과 적용과 사람의 완료 처리

여기서 중요한 점이 하나 있습니다. **보드 상태와 원본 적용은 별개입니다.** 보드 실행도 결과를 패치로 보관할 뿐, 원본은 바꾸지
않습니다. 그래서 순서는 이렇습니다.

1. 4장에서 본 run id로 `diff`를 검토하고 `apply`합니다.
2. 확인이 끝나면 사람이 보드에서 완료 처리합니다.

In [9]:
impl_run = re.search(r"planned → running\s+\[([0-9a-f-]{36})\]", shown).group(1)
print("보드 작업의 run id:", impl_run)
box.sm("diff", impl_run)
box.sm("apply", impl_run)
box.git("add", "textstats.py")
box.git("commit", "-qm", "add count_words")

보드 작업의 run id: ce3ec018-06d8-4a91-96aa-32385d8f0f89


diff --git a/textstats.py b/textstats.py
new file mode 100644
index 0000000..9a68e44
--- /dev/null
+++ b/textstats.py
@@ -0,0 +1,2 @@
+def count_words(text):
+    return len(text.split())


Applied run ce3ec018-06d8-4a91-96aa-32385d8f0f89
Receipt: /private/tmp/syncmill-notebook.ic8ogry9/state/runs/5a8bb136-069e-4da3-874c-dc42d84c290c/runs/ce3ec018-06d8-4a91-96aa-32385d8f0f89/applications/20261014T121523894303Z.json


<git commit: exit 0>

이제 완료 처리합니다. 먼저 **증거 없이** `done`으로 바꾸려고 하면 어떻게 될까요?

In [10]:
refused = box.sm("board", "update", impl_id, "--state", "done", check=False)
assert refused.returncode == 1

Error: cannot mark item cc49cf9f-6440-494b-a820-a5d49ea26985 done without verification — attach passing evidence (--evidence) or a human sign-off (--sign-off)
(exit code 1)


`cannot mark item … done without verification` — 사람도 **근거 없이** 완료 처리할 수는 없습니다. 증거를 붙이거나
(`--evidence`), 사람이 직접 확인했다는 표시(`--sign-off`)를 붙여야 합니다.

In [11]:
box.sm("board", "update", impl_id, "--state", "done", "--sign-off", show=False)
shown = box.sm("board", "show", impl_id).stdout
assert re.search(r"(?m)^state:\s+done", shown)

id:       cc49cf9f-6440-494b-a820-a5d49ea26985
state:    done
goal:     textstats
task:     textstats.py에 count_words(text)를 구현해 줘
owner:    codex
outputs:  textstats.py
handoff:  count_words는 split() 기반입니다
evidence:
  - [human/pass] sign-off (by human)
history:
  · → planned  [human] — created
  planned → running  [ce3ec018-06d8-4a91-96aa-32385d8f0f89] — claimed
  running → review  [codex] — agent requested done without verification — clamped to review
  review → done  [human]


- `evidence: [human/pass] sign-off (by human)` — 사람이 확인했다는 증거가 기록됐습니다.
- `history`의 `review → done [human]` — 누가 완료 처리했는지 남습니다.

## 6. 검사가 있으면 자동으로 완료

매번 사람이 완료 처리하는 것은 번거롭습니다. **품질 게이트**(01의 10장)를 설정해 두면, 에이전트가 `done`을 요청하고 **그 실행에서
게이트를 통과했을 때** 보드가 바로 `done`으로 기록합니다. 검사가 곧 증거가 됩니다.

In [12]:
(box.repo / ".gitignore").write_text("__pycache__/\n", encoding="utf-8")
(box.repo / "check_textstats.py").write_text(
    "from textstats import count_words\n\n"
    "assert count_words('a b  c') == 3\n"
    "print('check_textstats: all good')\n",
    encoding="utf-8",
)
toml_path.write_text(
    toml_path.read_text(encoding="utf-8").replace("commands = []", 'commands = ["python3 check_textstats.py"]'),
    encoding="utf-8",
)
box.git("add", ".gitignore", "check_textstats.py", "syncmill.toml")
box.git("commit", "-qm", "add a gate")

box.set_scenario({"codex": {
    "files": {"README.md": "# Notebook demo\n\n`count_words('a b')`는 2를 돌려줍니다.\n"},
    "message": "Added a usage example."
               + board_block(state="done", handoff="README에 예시 한 줄 추가", outputs=["README.md"]),
}})
box.sm("board", "run")


[codex] Completed in 0.03s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: README.md

Added a usage example.
```syncmill-board
{"state": "done", "handoff": "README에 예시 한 줄 추가", "outputs": ["README.md"]}
```


<syncmill board: exit 0>

In [13]:
shown = box.sm("board", "show", docs_id).stdout
assert re.search(r"(?m)^state:\s+done", shown)
assert "[gate/pass]" in shown

id:       ec54694f-1d25-4ce3-aaee-fe091a16c757
state:    done
goal:     textstats
task:     README에 count_words 사용 예시를 추가해 줘
owner:    codex
depends:  cc49cf9f-6440-494b-a820-a5d49ea26985
outputs:  README.md
handoff:  README에 예시 한 줄 추가
evidence:
  - [gate/pass] 1/1 checks passed (by codex)
history:
  · → planned  [human] — created
  planned → running  [ec350576-efc3-457e-81b9-1c82cbb9a626] — claimed
  running → done  [codex] — agent completed; verified


**출력 읽는 법**

- 이번에는 `--agent`를 주지 않았습니다. 선행 작업이 `done`이 됐으므로 두 번째 작업이 처리 가능한 다음 작업으로 골라졌습니다.
- `Quality gate: passed` — 결과가 검사를 통과했습니다.
- `state: done`, `evidence: [gate/pass] 1/1 checks passed (by codex)` — 사람의 확인 없이도 **검사 통과가 증거가 되어** 완료됐습니다.
- `history`의 `running → done [codex] — agent completed; verified` — "검증됨"이 이력에 남습니다.

> `done`이 됐어도 **원본에는 아직 들어가지 않았습니다.** 5장처럼 run id로 `diff` → `apply`하는 것은 여전히 여러분의 몫입니다.
> `runs list`에서 `pending`으로 남아 있는 결과가 있는지 확인하는 습관을 들이세요.

In [14]:
box.sm("runs", "list")

ec350576-efc3-457e-81b9-1c82cbb9a626  pending      codex      2026-10-14T12:15:25.096867+00:00
ce3ec018-06d8-4a91-96aa-32385d8f0f89  applied      codex      2026-10-14T12:15:21.390884+00:00


<syncmill runs: exit 0>

## 7. 실패한 작업: `blocked`

실패하면 어떻게 기록될까요? 새 작업을 하나 넣고 가짜 codex를 실패시켜 봅시다.

In [15]:
broken_item = item_id(box.sm("board", "add", "일부러 실패할 작업", show=False))
box.sm("board", "run", "--agent", "codex", check=False, extra_env={"FAKE_CODEX_FAIL": "1"})
box.sm("board", "list", "--state", "blocked")


[syncmill] Failed
All agents failed:
  [codex] codex (fake): simulated failure
[10/14/26 21:15:27] INFO     board: purged 1 stale board-run  supervisor.py:1307
                             namespace(s)                                       
(exit code 1)


  c7b9ece4-5440-418f-933a-6313b6be0c16  blocked   일부러 실패할 작업  (human-required)


<syncmill board: exit 0>

- 실패한 작업은 `blocked`가 되고 `(human-required)` — **사람 확인 필요** 표시가 붙습니다.
- 사람 확인 표시가 붙은 작업은 `board run`이 **자동으로 다시 집어 가지 않습니다.** 같은 실패를 계속 반복하지 않게 하기 위해서입니다.
- 원인을 해결한 뒤 `board update <id> --clear-human-required`로 표시를 지우고, 다시 실행할 수 있게 상태를 바꿉니다
  (TUI의 Board 탭에서는 `u`·`h` 키).

## 8. 팀 상태 지표: `board stats`

In [16]:
box.sm("board", "stats")

items: 3
  by state:  blocked=1  done=2
  reclaimed (stale-lease):  0
  redispatched (work redone):  0
  false-done (done, unverified):  0
  blocked > 3600s:  0
  human-required > 3600s:  0
  dropped handoffs > 3600s:  0


<syncmill board: exit 0>

**출력 읽는 법**

| 줄 | 뜻 | 0이 아니면 |
|---|---|---|
| `items` / `by state` | 작업 수와 상태별 개수 | |
| `reclaimed (stale-lease)` | 실행 도중 멈춘(프로세스가 죽은 등) 작업을 회수한 횟수 | 실행 환경이 불안정한지 확인 |
| `redispatched (work redone)` | 같은 작업을 두 번 이상 실행한 작업 수 | 중복 작업 비용이 생기고 있음 |
| `false-done (done, unverified)` | 증거 없이 `done`인 작업 수 | 정상 경로로는 생길 수 없음 — 무결성 경보 |
| `blocked > 3600s` 등 | 1시간 넘게 방치된 막힘·사람 확인 필요·인계 메모 | 누군가 봐야 할 작업이 쌓이고 있음 |

방치 기준 시간(기본 3600초)은 `[board]` `stall_seconds`로 바꿀 수 있습니다.

## 9. 같은 일을 화면으로: TUI

지금까지 한 일을 **터미널 화면**에서 키보드로 할 수도 있습니다. TUI는 실제 터미널(TTY)이 필요해서 이 노트북 안에서는 열 수
없습니다. 작업할 저장소에서 이렇게 엽니다.

```bash
syncmill tui              # 열 수 있는 첫 화면부터 (Rooms → Board → Runs → Status 순)
syncmill tui --tab board  # Board 탭부터
```

| 탭 | 하는 일 |
|---|---|
| `Board` | 이 노트북의 2~8장: 작업 추가(`a`), 결정 기록(`d`), 수정(`e`), 막힘·사람 확인 해제(`u`/`h`), 통계(`s`), 실행(`x`/`g`) |
| `Runs` | 실행 중인 작업의 상태·로그(Operations), 결과의 요약·diff와 적용(Artifacts) |
| `Reports` | 실행 기록(report) 보기 |
| `Teams` | 팀 프로필 보기·편집 |
| `Status` | 설정 확인 |
| `Sessions` / `Rooms` | 대화형 세션, 여러 CLI 후보 비교 |

공통 키: `1`~`7` 탭 이동, `r` 새로고침, `?` 도움말, `q` TUI 닫기(실행 중인 작업은 계속됨).

**처음 해 볼 순서** ([TUI 화면 안내서](../tui-guide.ko.md)의 권장 동선)

1. `Status`에서 설정 확인(실제 설치·로그인은 `doctor`)
2. `Board`에서 작고 확인 가능한 작업 하나 추가
3. `x`로 에이전트 하나를 골라 실행 — **실제 provider 호출이므로 비용이 듭니다**
4. `Runs / Operations`에서 진행과 로그 확인
5. `Runs / Artifacts`에서 요약과 diff 확인
6. 정확한 run을 확인한 뒤 `g`로 원본에 적용

## 10. 실무 팁

- **목표(goal)를 꼭 붙이세요.** `board run --goal <목표>`로 그 목표의 작업만 처리할 수 있고, 팀 프로필이 그 목표를 맡고 있으면
  그 팀의 에이전트·전략이 자동으로 적용됩니다.
- **`board run --loop`**은 처리할 수 있는 작업을 차례로 모두 실행합니다. 작업 수만큼 비용이 드니, 목록을 먼저 확인하세요.
- **게이트를 먼저 설정하세요.** 게이트가 없으면 모든 작업이 `review`에 쌓이고, 사람이 하나씩 완료 처리해야 합니다.
- **보드 `done` ≠ 원본 적용.** 완료 처리와 별개로 `runs list`의 `pending` 결과를 검토·적용하세요.
- **다른 도구에서 보드 읽기**: 03의 `syncmill serve`는 `board_list`·`board_get`·`board_stats` 읽기 전용 도구도 제공합니다.
- **결정은 짧고 분명하게.** 결정은 해당 목표의 모든 보드 실행 프롬프트에 통째로 들어갑니다.

## 11. 정리하기

In [17]:
box.set_scenario(None)
root = box.root
box.cleanup()
print("삭제됨:", not root.exists())

삭제됨: True


## 정리: 이번에 쓴 명령

| 명령 | 뜻 |
|---|---|
| `[board] enabled = true` (`syncmill.toml`) | 워크보드 켜기 |
| `syncmill board add "<할 일>" --goal <목표> [--depends-on <id>]` | 작업 등록 |
| `syncmill board list [--state <상태>] [--goal <목표>]` | 목록 |
| `syncmill board show <id>` | 작업 하나의 상태·증거·이력 |
| `syncmill board decide "<결정>" [--scope <목표>]` / `board decisions` | 팀 결정 기록 / 보기 |
| `syncmill board run [--id <id>] [--goal <목표>] [--loop]` | 다음 작업(들) 실행 |
| `syncmill board update <id> --state done --sign-off` | 사람이 확인하고 완료 처리 |
| `syncmill board update <id> --clear-human-required` | 사람 확인 표시 해제 |
| `syncmill board stats` | 팀 상태 지표 |
| `syncmill tui --tab board` | 같은 일을 터미널 화면에서 |

이것으로 시리즈를 마칩니다. 처음으로 돌아가 전체 그림을 다시 보고 싶다면 [00. 소개](00-introduction.ipynb)를, 명령과 옵션의 전체
목록은 [사용 안내서](../user-guide.ko.md)를 보세요.